In [1]:
import pandas as pd
import numpy as np
import GEOparse
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import pickle

print("Libraries loaded!")

/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Libraries loaded!


In [2]:
# load raw GEO data
gse = GEOparse.get_GEO(geo="GSE13355", destdir="./data")

# collect PP (psoriasis) and NN (healthy) samples
samples = {}
labels = {}
for name, sample in gse.gsms.items():
    title = sample.metadata["title"][0]
    if "PP" in title:
        samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        labels[name] = 1
    elif "NN" in title:
        samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        labels[name] = 0

df_raw = pd.DataFrame(samples).T
y = pd.Series(labels)

# clean: remove control probes, NaNs, low variance genes
df_clean = df_raw.loc[:, ~df_raw.columns.str.startswith("AFFX")]
df_clean = df_clean.dropna(axis=1)
variances = df_clean.var()
df_clean = df_clean.loc[:, variances > variances.quantile(0.80)]

print("Shape:", df_clean.shape)
print("Should be: (122, 10923)")

25-Jun-2026 18:55:32 DEBUG utils - Directory ./data already exists. Skipping.
25-Jun-2026 18:55:32 INFO GEOparse - File already exist: using local version.
25-Jun-2026 18:55:32 INFO GEOparse - Parsing ./data/GSE13355_family.soft.gz: 
25-Jun-2026 18:55:32 DEBUG GEOparse - DATABASE: GeoMiame
25-Jun-2026 18:55:32 DEBUG GEOparse - SERIES: GSE13355
25-Jun-2026 18:55:32 DEBUG GEOparse - PLATFORM: GPL570
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GSM337197
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GSM337198
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GSM337199
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GSM337200
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GSM337201
25-Jun-2026 18:55:33 DEBUG GEOparse - SAMPLE: GS

Shape: (122, 10923)
Should be: (122, 10923)


In [3]:
# split into groups
df_psoriasis = df_clean[y == 1]
df_healthy = df_clean[y == 0]

# t-test for every gene
pvalues = {}
tstatistics = {}
for gene in df_clean.columns:
    t_stat, p_val = stats.ttest_ind(df_psoriasis[gene], df_healthy[gene])
    pvalues[gene] = p_val
    tstatistics[gene] = t_stat

results = pd.DataFrame({'pvalue': pvalues, 'tstatistic': tstatistics})

# bonferroni correction, keep top 300
bonferroni_cutoff = 0.05 / len(results)
significant = results[results['pvalue'] < bonferroni_cutoff].sort_values('pvalue')
top_genes = significant.head(300).index.tolist()
df_filtered = df_clean[top_genes]

print("Top genes:", len(top_genes))
print("Filtered shape:", df_filtered.shape)

Top genes: 300
Filtered shape: (122, 300)


In [4]:
# load GSE14905
gse2 = GEOparse.get_GEO(geo="GSE14905", destdir="./data")

# collect Lesion Skin and Normal Skin samples
val_samples = {}
val_labels = {}
for name, sample in gse2.gsms.items():
    title = sample.metadata["title"][0]
    if title.startswith("Lesion Skin"):
        val_samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        val_labels[name] = 1
    elif title.startswith("Normal Skin"):
        val_samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        val_labels[name] = 0

df_val = pd.DataFrame(val_samples).T
y_val = pd.Series(val_labels)

# filter to our 300 genes
df_val_filtered = df_val[top_genes]

print("Validation samples:", df_val_filtered.shape[0])
print("Psoriasis:", sum(y_val == 1))
print("Healthy:", sum(y_val == 0))

25-Jun-2026 19:01:41 DEBUG utils - Directory ./data already exists. Skipping.
25-Jun-2026 19:01:41 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz to ./data/GSE14905_family.soft.gz
100%|█| 47.9M/47.9M [00:03<00:00, 13.1
25-Jun-2026 19:01:46 DEBUG downloader - Size validation passed
25-Jun-2026 19:01:46 DEBUG downloader - Moving /var/folders/7m/b89066rn4db_62r6g83zklmh0000gn/T/tmpti9rz06i to /Users/arthikadalabalumatha/Documents/psoraisis-ai/data/GSE14905_family.soft.gz
25-Jun-2026 19:01:46 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz
25-Jun-2026 19:01:46 INFO GEOparse - Parsing ./data/GSE14905_family.soft.gz: 
25-Jun-2026 19:01:46 DEBUG GEOparse - DATABASE: GeoMiame
25-Jun-2026 19:01:46 DEBUG GEOparse - SERIES: GSE14905
25-Jun-2026 19:01:46 DEBUG GEOparse - PLATFORM: GPL570
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python

Validation samples: 54
Psoriasis: 33
Healthy: 21


In [5]:
# combine training and validation data
df_combined = pd.concat([df_filtered, df_val_filtered])
y_combined = pd.concat([y, y_val])

print("Combined shape:", df_combined.shape)
print("Psoriasis:", sum(y_combined == 1))
print("Healthy:", sum(y_combined == 0))

# train model on raw values (no scaling - avoids batch effect)
model = LogisticRegression(max_iter=1000, random_state=42, C=0.1)
model.fit(df_combined, y_combined)

print("\nModel trained!")
print("Features:", model.n_features_in_)

Combined shape: (176, 300)
Psoriasis: 91
Healthy: 85

Model trained!
Features: 300


/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/arthikadalabalumatha/Documents/psoraisis-ai/.venv/lib/python3.9/site-packages/sk

In [6]:
def get_risk_label(prob):
    if prob < 0.20:
        return "Low risk of psoriasis"
    elif prob < 0.40:
        return "Slightly elevated risk of psoriasis"
    elif prob < 0.60:
        return "Moderate risk of psoriasis — further testing recommended"
    elif prob < 0.80:
        return "High risk of psoriasis"
    else:
        return "Very high risk of psoriasis"

def score_patient(patient_data):
    prob = model.predict_proba(patient_data)[0][1]
    return float(prob)

# test on all 54 validation samples
correct = 0
for i in range(54):
    patient = df_val_filtered.iloc[[i]]
    prob = score_patient(patient)
    predicted = 1 if prob >= 0.5 else 0
    actual = y_val.iloc[i]
    if predicted == actual:
        correct += 1
    label = get_risk_label(prob)
    print(f"Sample {i+1:2d}: {prob:.1%} — {label} (Actual: {'Psoriasis' if actual == 1 else 'Healthy'})")

print()
print(f"Correct: {correct}/54")
print(f"Accuracy: {correct/54:.1%}")

Sample  1: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample  2: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample  3: 0.0% — Low risk of psoriasis (Actual: Healthy)
Sample  4: 0.0% — Low risk of psoriasis (Actual: Healthy)
Sample  5: 0.0% — Low risk of psoriasis (Actual: Healthy)
Sample  6: 7.5% — Low risk of psoriasis (Actual: Healthy)
Sample  7: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample  8: 0.2% — Low risk of psoriasis (Actual: Healthy)
Sample  9: 0.8% — Low risk of psoriasis (Actual: Healthy)
Sample 10: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample 11: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample 12: 0.0% — Low risk of psoriasis (Actual: Healthy)
Sample 13: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample 14: 8.2% — Low risk of psoriasis (Actual: Healthy)
Sample 15: 0.2% — Low risk of psoriasis (Actual: Healthy)
Sample 16: 0.1% — Low risk of psoriasis (Actual: Healthy)
Sample 17: 0.4% — Low risk of psoriasis (Actual: Healthy)
Sample 18: 0.1

In [7]:
# save model and top genes
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("top_genes.pkl", "wb") as f:
    pickle.dump(top_genes, f)

# delete old scaler since we don't need it anymore
import os
if os.path.exists("scaler.pkl"):
    os.remove("scaler.pkl")

print("Saved!")
print("Model features:", model.n_features_in_)
print("Top genes:", len(top_genes))

Saved!
Model features: 300
Top genes: 300
